# Day 082 — Exercise 4: Recall — Injecting Memory into the Prompt

**What you'll build:** `build_memory_prompt` — assemble the chat prompt so the model *sees* what's remembered: the long-term profile up top, the recent turns next, the new message last.

**Why it matters:** storing memory is only half the job. Memory only helps if it's put back *in front of the model* at the right place. This is recall: the durable profile becomes system context, working memory supplies recent turns, and the new question follows.

In [ ]:

# ── short-term working memory (this session only) ────────────────────────────
class WorkingMemory:
    """Short-term memory: the recent turns of the current session.

    Bounded to the last `max_turns` messages so the prompt never grows without
    limit, and cleared at a session boundary. This is the agent's scratchpad -
    it does NOT survive a restart.
    """

    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self._turns = []

    def add(self, role, content):
        """Append a {role, content} turn; keep only the last max_turns. Returns self."""
        self._turns.append({"role": role, "content": str(content)})
        if len(self._turns) > self.max_turns:
            self._turns = self._turns[-self.max_turns:]
        return self

    def turns(self):
        """Return a copy of the recent turns."""
        return list(self._turns)

    def render(self):
        """Render the turns as text, one 'role: content' line each."""
        return "\n".join(t["role"] + ": " + t["content"] for t in self._turns)

    def clear(self):
        """Forget the session (end-of-session boundary)."""
        self._turns.clear()

    def __len__(self):
        return len(self._turns)

# ── long-term memory (persists across sessions, SQLite) ──────────────────────
import sqlite3


class LongTermMemory:
    """Durable key/value memory backed by SQLite - survives restarts.

    Default db_path is ":memory:" (a private in-process database). Pass a file
    path to persist across sessions: a new LongTermMemory on the same path sees
    everything a previous one remembered.
    """

    def __init__(self, db_path=":memory:"):
        self.db_path = db_path
        self._conn = sqlite3.connect(db_path)
        self._conn.execute(
            "CREATE TABLE IF NOT EXISTS memories "
            "(key TEXT PRIMARY KEY, value TEXT NOT NULL)"
        )
        self._conn.commit()

    def remember(self, key, value):
        """Store or overwrite a fact by key. Returns self."""
        self._conn.execute(
            "INSERT OR REPLACE INTO memories(key, value) VALUES(?, ?)",
            (str(key), str(value)))
        self._conn.commit()
        return self

    def recall(self, key):
        """Return the stored value for a key, or None if unknown."""
        row = self._conn.execute(
            "SELECT value FROM memories WHERE key = ?", (str(key),)).fetchone()
        return row[0] if row else None

    def search(self, term):
        """Return [{key, value}] where term (case-insensitive) is in key or value."""
        term = str(term).lower()
        rows = self._conn.execute("SELECT key, value FROM memories").fetchall()
        return [{"key": k, "value": v} for k, v in rows
                if term in k.lower() or term in v.lower()]

    def all(self):
        """Return every fact as [{key, value}], ordered by key."""
        rows = self._conn.execute(
            "SELECT key, value FROM memories ORDER BY key").fetchall()
        return [{"key": k, "value": v} for k, v in rows]

    def forget(self, key):
        """Delete a fact by key. Returns self."""
        self._conn.execute("DELETE FROM memories WHERE key = ?", (str(key),))
        self._conn.commit()
        return self

    def __len__(self):
        return self._conn.execute("SELECT COUNT(*) FROM memories").fetchone()[0]

    def close(self):
        """Close the database connection."""
        self._conn.close()


## Task

`build_memory_prompt(message, working, longterm) -> list[dict]`

1. `facts = longterm.all()`; render a profile of `- key: value` lines (use `"(nothing yet)"` when empty).
2. A `system` message: an assistant intro + `What you remember about the user:` + the profile.
3. Then `working.turns()`, then the new `{'role': 'user', 'content': message}` last.

## Your Implementation

In [ ]:
def build_memory_prompt(message, working, longterm):
    """Inject long-term profile + working-memory turns, then the new message."""
    raise NotImplementedError


In [ ]:

# ── recall: injecting memory into the prompt ─────────────────────────────────
def build_memory_prompt(message, working, longterm):
    """Build a chat prompt that injects long-term profile + short-term turns.

    The system message carries what we durably know about the user; the recent
    working-memory turns follow; the new message comes last.
    """
    facts = longterm.all()
    profile = "\n".join("- " + f["key"] + ": " + f["value"] for f in facts)
    system = "\n".join([
        "You are a helpful assistant with memory of the user.",
        "",
        "What you remember about the user:",
        profile if profile else "(nothing yet)",
    ])
    messages = [{"role": "system", "content": system}]
    messages.extend(working.turns())
    messages.append({"role": "user", "content": str(message)})
    return messages


## Automated checks

In [ ]:

score, total = 0, 4
try:
    wm = WorkingMemory()
    lt = LongTermMemory()
    lt.remember('name', 'Kutlwano')

    prompt = build_memory_prompt('hello', wm, lt)
    assert prompt[0]['role'] == 'system' and 'name: Kutlwano' in prompt[0]['content']
    score += 1; print("✅ long-term facts are injected into the system message")

    assert prompt[-1] == {'role': 'user', 'content': 'hello'}
    score += 1; print("✅ the new message comes last")

    wm.add('user', 'earlier').add('assistant', 'noted')
    p2 = build_memory_prompt('now', wm, lt)
    assert {'role': 'assistant', 'content': 'noted'} in p2 and p2[-1]['content'] == 'now'
    score += 1; print("✅ working-memory turns sit between system and new message")

    empty = build_memory_prompt('hi', WorkingMemory(), LongTermMemory())
    assert 'nothing yet' in empty[0]['content']
    score += 1; print("✅ an empty profile still builds a valid prompt")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── recall: injecting memory into the prompt ─────────────────────────────────
def build_memory_prompt(message, working, longterm):
    """Build a chat prompt that injects long-term profile + short-term turns.

    The system message carries what we durably know about the user; the recent
    working-memory turns follow; the new message comes last.
    """
    facts = longterm.all()
    profile = "\n".join("- " + f["key"] + ": " + f["value"] for f in facts)
    system = "\n".join([
        "You are a helpful assistant with memory of the user.",
        "",
        "What you remember about the user:",
        profile if profile else "(nothing yet)",
    ])
    messages = [{"role": "system", "content": system}]
    messages.extend(working.turns())
    messages.append({"role": "user", "content": str(message)})
    return messages
```

**Why put the profile in the *system* message?** The durable facts are standing context that should color every reply, not a one-off user turn. System is where persistent instructions live, so the profile shapes the whole conversation rather than being buried mid-history.

</details>